# Bivariate maps

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import pyarrow.feather as arrow_feather

HERE = Path.cwd()
EXPECTED_RESULT = '2d6ed055b14567bd6cdbc112c35a1cdcb32c5430eafabf2f0f7f8c4771b59d70'
EXPECTED_MANIFEST_SHA256 = '8f779deafdb892fedc2503aeb20a4a9d1623b7d56bf2df639402677e1a5a5b80'
EXPECTED_PAYLOAD_SHA256 = '8613750a2d50e3523855826ab02bad073c2e51b8c0716d70cdcda4bb86d9aadf'
METRICS = ['fd_int', 'ann_gshrt', 'seas_mis', 'seas_def', 'diur_mis', 'eve_def']
CLASSES = ['fd_cls', 'ag_cls', 'sd_cls', 'sl_cls', 'ma_cls', 'hl_cls']

def payload_sha256(frame):
    pass
    columns = ['province_index', 'province', 'hour_index', 'demand_facility_mw',
               'accessible_green_mw', 'result_sha256', 'input_version', 'geometry_wkb']
    assert list(frame.columns) == columns
    digest = hashlib.sha256()
    digest.update(str(len(frame)).encode('ascii'))
    for column in columns:
        digest.update(column.encode('ascii') + b'\0')
        if column in ['province_index', 'hour_index']:
            digest.update(np.asarray(frame[column], dtype='<i8').tobytes())
        elif column in ['demand_facility_mw', 'accessible_green_mw']:
            digest.update(np.asarray(frame[column], dtype='<f8').tobytes())
        elif column == 'geometry_wkb':
            for value in frame[column]:
                if value is None:
                    digest.update(b'\0')
                else:
                    digest.update(b'\1' + len(value).to_bytes(8, 'little') + value)
        else:
            values = frame[column].tolist()
            assert all(isinstance(value, str) and '\0' not in value for value in values)
            digest.update('\0'.join(values).encode('utf-8'))
    return digest.hexdigest()

def month_slices(month_days):
    slices = []
    start = 0
    for days in month_days:
        stop = start + int(days)
        slices.append(slice(start, stop))
        start = stop
    return slices

def normalized_overlap(left, right, axis):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    left = left / np.maximum(left.sum(axis=axis, keepdims=True), 1e-12)
    right = right / np.maximum(right.sum(axis=axis, keepdims=True), 1e-12)
    return np.minimum(left, right).sum(axis=axis)

def strict_terciles(values):
    ranks = values.rank(method='first')
    return pd.qcut(ranks, 3, labels=[1, 2, 3]).astype(int).to_numpy()

def load_inputs():
    path = HERE / 'data/bivariate_mismatch_inputs_corrected.feather'
    assert path.stat().st_size < 100 * 1024**2
    table = arrow_feather.read_table(path)
    raw_manifest = (table.schema.metadata or {}).get(b'compute_power_bivariate_manifest')
    if raw_manifest is None:
        raise ValueError('This is the older sidecar-dependent Feather. Replace both the notebook and Feather with the single-file version.')
    assert hashlib.sha256(raw_manifest).hexdigest() == EXPECTED_MANIFEST_SHA256, 'Input manifest changed'
    manifest = json.loads(raw_manifest.decode('utf-8'))
    frame = table.to_pandas()
    assert payload_sha256(frame) == manifest['payload_sha256'] == EXPECTED_PAYLOAD_SHA256, 'Input data changed'
    manifest['feather_sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
    for column, expected in [('result_sha256', EXPECTED_RESULT), ('input_version', 'audit_corrected_20261007_configuration_01')]:
        assert frame[column].notna().all() and set(frame[column]) == {expected}, column
    assert manifest['formal_result_sha256'] == EXPECTED_RESULT
    assert manifest['configuration_kwargs'] == {'flexible_shift_share': 0.0, 'green_access_multiplier': 3.0,
        'we_percentile': 50, 'it_demand_anchor_we_percentile': 50}
    frame = frame.sort_values(['province_index', 'hour_index'], kind='stable')
    assert not frame.duplicated(['province_index', 'hour_index']).any()
    assert len(frame) == 31 * 8760
    provinces = frame.groupby('province_index', sort=True).province.first().tolist()
    assert len(set(provinces)) == 31
    assert np.array_equal(frame.hour_index.to_numpy(), np.tile(np.arange(8760), 31))
    assert np.array_equal(frame.province_index.to_numpy(), np.repeat(np.arange(31), 8760))
    numeric = frame[['demand_facility_mw', 'accessible_green_mw']].to_numpy(dtype=float)
    assert np.isfinite(numeric).all() and (numeric >= 0).all()
    data = SimpleNamespace(provinces=provinces, month_days=np.array(manifest['month_days']),
        demand_facility_hourly=numeric[:, 0].reshape(31, 365, 24),
        accessible_green_hourly=numeric[:, 1].reshape(31, 365, 24))
    shapes = frame[frame.geometry_wkb.notna()][['province', 'geometry_wkb']]
    assert len(shapes) == 31 and shapes.province.is_unique
    geometry = gpd.GeoDataFrame(shapes[['province']].copy(),
        geometry=gpd.GeoSeries.from_wkb(shapes.geometry_wkb, index=shapes.index, crs=manifest['geometry_crs']))
    assert not geometry.geometry.is_empty.any() and geometry.geometry.notna().all()
    return data, geometry, manifest

def calculate_metrics(data):
    slices = month_slices(data.month_days)
    annual_demand = data.demand_facility_hourly.sum(axis=(1, 2)) / 1e6
    annual_green = data.accessible_green_hourly.sum(axis=(1, 2)) / 1e6
    annual_shortage = np.clip(1 - annual_green / np.maximum(annual_demand, 1e-9), 0, None)
    demand_month = np.stack([data.demand_facility_hourly[:, item, :].sum(axis=(1, 2)) / 1e6 for item in slices], axis=1)
    green_month = np.stack([data.accessible_green_hourly[:, item, :].sum(axis=(1, 2)) / 1e6 for item in slices], axis=1)
    seasonal_mismatch = 1 - normalized_overlap(demand_month, green_month, axis=1)
    seasonal_shortage = np.maximum(demand_month - green_month, 0).sum(axis=1) / np.maximum(demand_month.sum(axis=1), 1e-9)
    demand_hour = data.demand_facility_hourly.mean(axis=1)
    green_hour = data.accessible_green_hourly.mean(axis=1)
    diurnal_mismatch = 1 - normalized_overlap(demand_hour, green_hour, axis=1)
    evening = [18, 19, 20, 21]
    high_load_shortage = np.maximum(data.demand_facility_hourly[:, :, evening] - data.accessible_green_hourly[:, :, evening], 0).sum(axis=(1, 2)) / np.maximum(data.demand_facility_hourly.sum(axis=(1, 2)), 1e-9)
    metrics = pd.DataFrame({'province': data.provinces, 'fd_int': annual_demand, 'ann_gshrt': annual_shortage,
        'seas_mis': seasonal_mismatch, 'seas_def': seasonal_shortage, 'diur_mis': diurnal_mismatch, 'eve_def': high_load_shortage})
    for metric, class_name in zip(METRICS, CLASSES):
        metrics[class_name] = strict_terciles(metrics[metric])
    return metrics

def main():
    data, geometry, manifest = load_inputs()
    metrics = calculate_metrics(data)
    assert np.isfinite(metrics[METRICS].to_numpy()).all()
    assert (metrics[METRICS[1:]].to_numpy() >= -1e-12).all()
    assert (metrics[METRICS[1:]].to_numpy() <= 1 + 1e-12).all()
    assert all(set(metrics[column]) == {1, 2, 3} for column in CLASSES)
    reference = pd.DataFrame(manifest['current_atlas_annual_reference']).set_index('province').loc[data.provinces]
    annual_green = data.accessible_green_hourly.sum(axis=(1, 2)) / 1e6
    errors = {'demand_twh': float(np.max(np.abs(metrics.fd_int.to_numpy() - reference.facility_demand_twh.to_numpy()))),
              'green_twh': float(np.max(np.abs(annual_green - reference.accessible_green_twh.to_numpy())))}
    assert max(errors.values()) < 1e-8, errors
    merged = geometry.merge(metrics, on='province', how='inner', validate='one_to_one')
    assert len(merged) == 31
    merged['result_sha'] = EXPECTED_RESULT[:16]
    metric_fields = ['province', *METRICS, *CLASSES, 'result_sha']
    class_fields = ['province', *CLASSES, 'result_sha']
    out = HERE / 'bivariate_mismatch_gis'
    out.mkdir(exist_ok=True)
    for name, fields in [('bivariate_mismatch_metrics', metric_fields), ('bivariate_mismatch_classes', class_fields)]:
        layer = gpd.GeoDataFrame(merged[fields + ['geometry']], geometry='geometry', crs=geometry.crs)
        layer.to_file(out / (name + '.shp'), encoding='utf-8', index=False)
        layer.to_file(out / (name + '.gpkg'), layer=name, driver='GPKG', index=False)
        layer.drop(columns='geometry').to_csv(out / (name + '.csv'), index=False, encoding='utf-8-sig')
        recovered = gpd.read_file(out / (name + '.shp'))
        assert list(recovered.columns) == fields + ['geometry']
        assert recovered.crs == layer.crs
        recovered = recovered.set_index('province').loc[layer.province]
        assert np.array_equal(recovered.geometry.to_wkb().to_numpy(), layer.geometry.to_wkb().to_numpy())
        assert set(recovered.result_sha) == {EXPECTED_RESULT[:16]}
        for field in fields:
            if field in METRICS:
                np.testing.assert_allclose(recovered[field], layer[field], atol=1e-12, rtol=1e-12)
            elif field in CLASSES:
                assert np.array_equal(recovered[field], layer[field])
    report = {'status': 'PASS', 'province_count': 31, 'formal_result_sha256': EXPECTED_RESULT,
        'frozen_input_sha256': manifest['frozen_input_sha256'], 'feather_sha256': manifest['feather_sha256'],
        'metric_fields': metric_fields, 'class_fields': class_fields, 'atlas_errors': errors,
        'classification': 'Original strict rank-based terciles; method=first in original input province order',
        'panel_pairs_from_author_screenshot': {'A': ['fd_cls', 'ag_cls'], 'G': ['ma_cls', 'hl_cls'], 'H': ['sd_cls', 'sl_cls']},
        'boundary': 'Input diagnostic metrics, not optimization outcomes. Geometry only inherited; no old metric values read. No optimization or cooling regeneration.'}
    (out / 'provenance.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    pass
    return report

if __name__ == '__main__':
    main()
